# 02 — Clean the data

**Project:** Do U.S. climate change concerns move European green-minus-brown returns?
**Author:** Rafael Machado Cerqueira — Advanced Data Analytics, HEC Lausanne, Autumn 2026

**Input:** the raw files saved by `01_get_data.ipynb` in `data/raw/`.
**Output:** analysis-ready panels in `data/processed/`, used by `03_model.ipynb`.

| Step | What it does |
|---|---|
| 1 | Load the raw files and cut the sample to January 2010 – June 2025 |
| 2 | Fix units (London prices in pence) and convert European prices to EUR |
| 3 | Build the European and U.S. trading calendars |
| 4 | Compute daily returns |
| 5 | Liquidity rule: drop the years in which a stock's price is frozen too often |
| 6 | Build the green, brown and green-minus-brown portfolios |
| 7 | Map the daily MCCC index (all calendar days) to trading days |
| 8 | Align the controls (VIX, Brent, Fama-French) |
| 9 | Assemble the European and U.S. panels, check them and save |

Every choice is a parameter in the setup cell and is written to `cleaning_log.json`, so the paper can report it.

*Code drafted with Claude (Anthropic) and reviewed by the author — declared in the paper's appendix.*

## 0. Setup and parameters

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

# ---- Cleaning choices (each one is reported in the paper) ----
START, END = "2010-01-01", "2025-06-30"  # END = last day covered by the MCCC 2025 vintage
LIQ_THRESHOLD = 0.10          # max share of zero-return days per year for a stock to count as traded
LIQ_THRESHOLD_ROBUST = 0.05   # robustness check
MIN_EU_COVERAGE = 0.50        # a day is a European trading day if at least 50% of listed stocks trade
MIN_STOCKS_PER_SIDE = 3       # a portfolio return needs at least 3 stocks that day
WEEKEND_RULE = "mean"         # how non-trading days of MCCC are folded into the next trading day
FX_FFILL_LIMIT = 5            # max days an exchange rate is carried forward

# Corporate actions that Yahoo's adjusted prices do not handle correctly:
# the stock's returns are set to missing over the whole window (start, end), both included
CORPORATE_ACTION_WINDOWS = {
    "HOLN.SW": [("2025-06-16", "2025-06-30")],  # Amrize spin-off: 1 Amrize share per Holcim share,
                                                # Amrize traded from 23 June 2025; Yahoo's series is
                                                # unreliable over this window (-48% on 16 June, +14% on 23 June)
}

log = {"parameters": dict(START=START, END=END, LIQ_THRESHOLD=LIQ_THRESHOLD,
                          LIQ_THRESHOLD_ROBUST=LIQ_THRESHOLD_ROBUST,
                          MIN_EU_COVERAGE=MIN_EU_COVERAGE,
                          MIN_STOCKS_PER_SIDE=MIN_STOCKS_PER_SIDE,
                          WEEKEND_RULE=WEEKEND_RULE, FX_FFILL_LIMIT=FX_FFILL_LIMIT,
                          CORPORATE_ACTION_WINDOWS=CORPORATE_ACTION_WINDOWS)}
print("Reading from:", RAW)
print("Writing to:  ", PROC)

Reading from: /Users/RafaelMachadoCerqueira/Desktop/Data/climate-concerns-green-brown/data/raw
Writing to:   /Users/RafaelMachadoCerqueira/Desktop/Data/climate-concerns-green-brown/data/processed


## 1. Load the raw data

Prices, controls and factors are cut to the sample. The MCCC index is kept from 2003: its earlier history is needed in `03_model` to estimate the model of *expected* concern before the sample starts.

In [2]:
universe = pd.read_csv(RAW / "universe.csv").set_index("ticker")
px = pd.read_csv(RAW / "prices_daily.csv", index_col=0, parse_dates=True).sort_index().loc[START:END]

mccc = pd.read_csv(RAW / "mccc_daily.csv")
mccc["Date"] = pd.to_datetime(mccc["Date"])
mccc = mccc.set_index("Date").sort_index().loc[:END]

fred = pd.read_csv(RAW / "fred_controls_daily.csv", index_col=0, parse_dates=True).sort_index().loc[START:END]
ff_us = pd.read_csv(RAW / "ff3_us_daily.csv", index_col=0, parse_dates=True).sort_index().loc[START:END] / 100
ff_eu = pd.read_csv(RAW / "ff3_europe_daily.csv", index_col=0, parse_dates=True).sort_index().loc[START:END] / 100
# Fama-French files are in percent -> divided by 100 to match the returns below

GREEN = universe.index[universe["group"] == "green_eu"].tolist()
BROWN = universe.index[universe["group"] == "brown_eu"].tolist()
STOCKS = GREEN + BROWN
US = ["ICLN", "XLE", "SPY"]

print(f"Prices: {px.shape}, {px.index.min().date()} -> {px.index.max().date()}")
print(f"MCCC:   {mccc.shape}, {mccc.index.min().date()} -> {mccc.index.max().date()}")
print(f"{len(GREEN)} green, {len(BROWN)} brown European stocks")
log["n_green"], log["n_brown"] = len(GREEN), len(BROWN)

Prices: (4040, 37), 2010-01-01 -> 2025-06-30
MCCC:   (8217, 35), 2003-01-01 -> 2025-06-30
13 green, 16 brown European stocks


## 2. Units and currencies

* London listings (Shell, BP) are quoted in **pence**: divided by 100.
* European prices are converted to **EUR** with the EUR/xxx rates from Yahoo (a price in GBP is divided by EUR/GBP).
  Returns in EUR are the view of a euro-based investor; returns in local currency are kept as a robustness check.
* Exchange rates are carried forward for at most `FX_FFILL_LIMIT` days when Yahoo has no quote that day.

In [3]:
ccy = universe["currency"]
px_local = px[STOCKS].copy()

pence = [t for t in STOCKS if ccy[t] == "GBp"]
px_local[pence] = px_local[pence] / 100
ccy_major = ccy.replace({"GBp": "GBP"})

FX_COL = {"GBP": "EURGBP=X", "DKK": "EURDKK=X", "NOK": "EURNOK=X", "CHF": "EURCHF=X", "USD": "EURUSD=X"}
missing_fx = sorted(set(ccy_major[STOCKS]) - set(FX_COL) - {"EUR"})
assert not missing_fx, f"No exchange rate for: {missing_fx}"

px_eur = px_local.copy()
for t in STOCKS:
    c = ccy_major[t]
    if c != "EUR":
        rate = px[FX_COL[c]].ffill(limit=FX_FFILL_LIMIT)
        px_eur[t] = px_local[t] / rate

print("Divided by 100 (pence):", pence)
print(ccy_major[STOCKS].value_counts().to_string())
log["pence_tickers"] = pence

Divided by 100 (pence): ['SHEL.L', 'BP.L']
currency
EUR    22
DKK     2
NOK     2
GBP     2
CHF     1


## 3. Trading calendars

European exchanges have different holidays, so a date is kept as a **European trading day** if at least `MIN_EU_COVERAGE` of the stocks already listed have a price that day. The **U.S. trading days** are the days on which SPY trades.

In [4]:
listed = px_local.notna().cummax()                  # True once a stock has started trading
coverage = px_local.notna().sum(axis=1) / listed.sum(axis=1)
EU_DAYS = coverage.index[coverage >= MIN_EU_COVERAGE]
US_DAYS = px["SPY"].dropna().index

print(f"European trading days: {len(EU_DAYS)}  (dropped {len(coverage) - len(EU_DAYS)} dates with low coverage)")
print(f"U.S. trading days:      {len(US_DAYS)}")
log["n_eu_days"], log["n_us_days"] = len(EU_DAYS), len(US_DAYS)
log["n_dates_dropped_low_coverage"] = int(len(coverage) - len(EU_DAYS))

European trading days: 3948  (dropped 92 dates with low coverage)
U.S. trading days:      3896


## 4. Daily returns

Each stock's return is computed **between its own consecutive prices on the trading calendar**. After a local holiday, the next return covers two days instead of being lost. A price missing in the middle of the series is not filled in.

In [5]:
def own_day_returns(prices):
    """Simple returns between each column's consecutive non-missing prices."""
    return pd.concat({c: prices[c].dropna().pct_change() for c in prices.columns}, axis=1, sort=True).reindex(prices.index)

ret_eur = own_day_returns(px_eur.loc[EU_DAYS])
ret_local = own_day_returns(px_local.loc[EU_DAYS])
ret_us = own_day_returns(px.loc[US_DAYS, US])

summary = pd.DataFrame({
    "first_date": ret_eur.apply(lambda s: s.first_valid_index()),
    "n_returns": ret_eur.notna().sum(),
    "mean_daily_%": (ret_eur.mean() * 100).round(3),
    "vol_daily_%": (ret_eur.std() * 100).round(2),
    "side": universe.loc[STOCKS, "group"],
})
summary.sort_values(["side", "first_date"])

,first_date,n_returns,mean_daily_%,vol_daily_%,side
SHEL.L,2010-01-05,3884,0.024,1.75,brown_eu
BP.L,2010-01-05,3883,0.006,1.91,brown_eu
TTE.PA,2010-01-05,3947,0.039,1.61,brown_eu
ENI.MI,2010-01-05,3934,0.032,1.65,brown_eu
EQNR.OL,2010-01-05,3888,0.044,1.91,brown_eu
REP.MC,2010-01-05,3946,0.033,1.96,brown_eu
OMV.VI,2010-01-05,3876,0.052,1.97,brown_eu
GALP.LS,2010-01-05,3947,0.038,1.92,brown_eu
HEI.DE,2010-01-05,3934,0.063,1.96,brown_eu
HOLN.SW,2010-01-05,3892,0.036,1.85,brown_eu


**Check for extreme returns.** A daily move above 40% in absolute value is rare for these companies. Each case listed below should be explained (rights issue, crisis day, data error) before it is kept.

Findings: Solaria's two jumps in June 2012 fall in its illiquid years and are excluded by the liquidity rule anyway. Holcim's −48% on 16 June 2025 is not a market move: Holcim spun off its North American business, Amrize (one Amrize share per Holcim share held at the close of 20 June; Amrize traded from 23 June 2025). Yahoo's series records the drop a week early, then shows +14% on 23 June, the real ex-date. Because Holcim's prices are unreliable around the spin-off, its returns from 16 to 30 June 2025 are removed via `CORPORATE_ACTION_WINDOWS` (about 11 trading days of one brown stock out of 16).

In [6]:
extreme = ret_eur.stack()
extreme = extreme[extreme.abs() > 0.40].rename("return").reset_index()
extreme.columns = ["date", "ticker", "return"]
log["n_extreme_returns_over_40pct"] = len(extreme)

# Remove the returns affected by corporate actions (windows listed in the setup cell)
n_removed = 0
for t, windows in CORPORATE_ACTION_WINDOWS.items():
    for start, end in windows:
        mask = (ret_eur.index >= start) & (ret_eur.index <= end)
        n_removed += int(ret_eur.loc[mask, t].notna().sum())
        ret_eur.loc[mask, t] = np.nan
        ret_local.loc[mask, t] = np.nan
log["n_returns_removed_corporate_actions"] = n_removed
print(f"Returns set to missing (corporate actions): {n_removed}", CORPORATE_ACTION_WINDOWS)
extreme

Returns set to missing (corporate actions): 11 {'HOLN.SW': [('2025-06-16', '2025-06-30')]}


,date,ticker,return
0,2012-06-05,SLR.MC,0.536232
1,2012-06-06,SLR.MC,0.433962
2,2025-06-16,HOLN.SW,-0.482598


## 5. Liquidity rule (frozen prices)

A liquid stock almost never closes exactly unchanged. For each stock and year, the share of **zero-return days** is computed on local-currency prices (FX conversion would hide frozen prices).

**Rule:** a stock enters the portfolio on 1 January of the first year **from which** this share stays below `LIQ_THRESHOLD` in every later year. Returns before that date are set to missing. The same rule with `LIQ_THRESHOLD_ROBUST` gives the robustness version.

In [7]:
zero_share = (ret_local == 0).where(ret_local.notna()).groupby(ret_local.index.year).mean()

def entry_years(zero_share, threshold):
    """First year from which the zero-return share stays below the threshold (None = never)."""
    out = {}
    for t in zero_share.columns:
        s = zero_share[t].dropna()
        bad = s.index[s >= threshold]
        if len(bad) == 0:
            out[t] = s.index.min()
        else:
            later = s.index[s.index > bad.max()]
            out[t] = later.min() if len(later) else None
    return pd.Series(out, dtype="object")

def apply_entry(rets, years):
    r = rets.copy()
    for t, y in years.items():
        if y is None or pd.isna(y):
            r[t] = np.nan
        else:
            r.loc[r.index < pd.Timestamp(int(y), 1, 1), t] = np.nan
    return r

entry_main = entry_years(zero_share, LIQ_THRESHOLD)
entry_robust = entry_years(zero_share, LIQ_THRESHOLD_ROBUST)

entry = pd.DataFrame({
    "side": universe.loc[STOCKS, "group"],
    "first_price": px_local.apply(lambda s: s.first_valid_index().date()),
    "max_zero_share": zero_share.max().round(3),
    f"entry_year_{int(LIQ_THRESHOLD*100)}pct": entry_main,
    f"entry_year_{int(LIQ_THRESHOLD_ROBUST*100)}pct": entry_robust,
})
log["entry_years_main"] = {t: (None if y is None else int(y)) for t, y in entry_main.items()}
entry.sort_values("max_zero_share", ascending=False)

,side,first_price,max_zero_share,entry_year_10pct,entry_year_5pct
GRE.MC,green_eu,2015-07-08,0.661,2020,2021
VLTSA.PA,green_eu,2010-01-04,0.344,2020,2022
SCATC.OL,green_eu,2014-10-02,0.133,2015,2019
SLR.MC,green_eu,2010-01-04,0.13,2017,2018
EDPR.LS,green_eu,2010-01-04,0.095,2010,2020
VER.VI,green_eu,2010-01-04,0.056,2010,2019
VOE.VI,brown_eu,2010-01-04,0.044,2010,2010
ERG.MI,green_eu,2010-01-04,0.043,2010,2010
HOLN.SW,brown_eu,2010-01-04,0.04,2010,2010
REP.MC,brown_eu,2010-01-04,0.039,2010,2010


In [8]:
# Zero-return share by year for the stocks the rule delays (the table shown in the paper)
delayed = entry.index[entry[f"entry_year_{int(LIQ_THRESHOLD*100)}pct"].astype(float)
                      > pd.Series([d.year for d in entry["first_price"]], index=entry.index)]
zero_share[delayed].round(3)

,SLR.MC,VLTSA.PA,SCATC.OL,GRE.MC
Date,,,,
2010,0.078431,0.239216,NaN,NaN
2011,0.042802,0.217899,NaN,NaN
2012,0.129921,0.322835,NaN,NaN
2013,0.118577,0.343874,NaN,NaN
2014,0.086614,0.232283,0.133333,NaN
2015,0.079051,0.059289,0.095618,0.634146
2016,0.11284,0.097276,0.027668,0.661479
2017,0.07451,0.086275,0.09127,0.647059
2018,0.023715,0.066929,0.068273,0.268775


## 6. Portfolios and spreads

* **Europe:** equal-weighted average of the green stocks minus that of the brown stocks, using every stock that has a return that day (after the liquidity rule). A side needs at least `MIN_STOCKS_PER_SIDE` stocks.
* **United States:** ICLN (clean energy) minus XLE (energy), in USD — the benchmark replication.

Three European versions are saved: main (EUR, 10% liquidity rule), liquidity robustness (5% rule), and local currency.

In [9]:
def side_return(rets, members):
    r = rets[members]
    n = r.notna().sum(axis=1)
    return r.mean(axis=1).where(n >= MIN_STOCKS_PER_SIDE), n

def spread(rets):
    g, ng = side_return(rets, GREEN)
    b, nb = side_return(rets, BROWN)
    return g, b, g - b, ng, nb

green_eu, brown_eu, spread_eu, n_green, n_brown = spread(apply_entry(ret_eur, entry_main))
_, _, spread_eu_liq_robust, _, _ = spread(apply_entry(ret_eur, entry_robust))
_, _, spread_eu_local, _, _ = spread(apply_entry(ret_local, entry_main))

spread_us = ret_us["ICLN"] - ret_us["XLE"]

n_by_year = pd.DataFrame({"green": n_green, "brown": n_brown}).groupby(EU_DAYS.year).median()
print("Median number of stocks per day, by year:")
n_by_year

Median number of stocks per day, by year:


,green,brown
Date,,
2010,8.0,16.0
2011,8.0,16.0
2012,8.0,16.0
2013,8.0,16.0
2014,8.0,16.0
2015,9.0,16.0
2016,10.0,16.0
2017,11.0,16.0
2018,11.0,16.0


In [10]:
def describe(s):
    s = s.dropna()
    return pd.Series({"start": s.index.min().date(), "days": len(s),
                      "mean_daily_%": round(s.mean() * 100, 4), "vol_daily_%": round(s.std() * 100, 3),
                      "ann_mean_%": round(s.mean() * 252 * 100, 2), "ann_vol_%": round(s.std() * np.sqrt(252) * 100, 2)})

pd.DataFrame({"green EU": describe(green_eu), "brown EU": describe(brown_eu),
              "spread EU": describe(spread_eu), f"spread EU ({int(LIQ_THRESHOLD_ROBUST*100)}% rule)": describe(spread_eu_liq_robust),
              "spread EU (local ccy)": describe(spread_eu_local), "spread US": describe(spread_us)}).T

,start,days,mean_daily_%,vol_daily_%,ann_mean_%,ann_vol_%
green EU,2010-01-05,3947,0.0558,1.529,14.06,24.28
brown EU,2010-01-05,3947,0.0318,1.413,8.01,22.43
spread EU,2010-01-05,3947,0.024,1.355,6.05,21.5
spread EU (5% rule),2010-01-05,3946,0.0244,1.432,6.15,22.74
spread EU (local ccy),2010-01-05,3947,0.0251,1.354,6.31,21.49
spread US,2010-01-05,3895,-0.0285,1.676,-7.19,26.61


## 7. MCCC index on trading days

The index has a value for **every calendar day**, weekends included, while prices exist only on trading days. Each calendar day is assigned to the **next trading day on or after it**; the values falling on the same trading day (e.g. Saturday, Sunday and Monday) are combined with `WEEKEND_RULE`.

Before the first market day (2003–2009), business days are used as the calendar, so that `03_model` can estimate expected concern on the pre-sample.

Days with a value of exactly 0 (no climate article) are kept: they mean *no concern in the news*, not missing data.

In [11]:
def mccc_on_calendar(index_df, market_days, rule):
    days = pd.DatetimeIndex(market_days).sort_values()
    pre = pd.bdate_range(index_df.index.min(), days.min() - pd.Timedelta(days=1))
    cal = pre.append(days)
    pos = cal.searchsorted(index_df.index, side="left")
    keep = pos < len(cal)
    m = index_df[keep].copy()
    m["trade_date"] = cal[pos[keep]]
    out = m.groupby("trade_date").agg(rule)
    out["n_calendar_days"] = m.groupby("trade_date").size()
    out.index.name = "date"
    return out.add_prefix("mccc_")

mccc_eu = mccc_on_calendar(mccc, EU_DAYS, WEEKEND_RULE)
mccc_us = mccc_on_calendar(mccc, US_DAYS, WEEKEND_RULE)

log["mccc_zero_days"] = int((mccc["Aggregate"] == 0).sum())
print(f"MCCC calendar days with value 0: {log['mccc_zero_days']}")
print("Calendar days per European trading day:")
print(mccc_eu.loc[START:, "mccc_n_calendar_days"].value_counts().sort_index().to_string())

MCCC calendar days with value 0: 547
Calendar days per European trading day:
mccc_n_calendar_days
1    3125
2       8
3     773
4      17
5      24
6       2


## 8. Controls

* **VIX:** daily change in points, between consecutive observations on each calendar.
* **Brent:** daily simple return of the spot price, same method.
* **Fama-French:** daily factors (U.S. factors on U.S. days, European factors on European days), in decimals.

Timing note for `03_model`: the VIX and U.S. factors are measured at the U.S. close, after European markets have closed.

In [12]:
def changes_on_calendar(series, days, kind):
    s = series.reindex(days).dropna()
    out = s.pct_change() if kind == "return" else s.diff()
    return out.reindex(days)

def controls(days, ff):
    ctrl = pd.DataFrame(index=days)
    ctrl["vix"] = fred["VIXCLS"].reindex(days)
    ctrl["vix_chg"] = changes_on_calendar(fred["VIXCLS"], days, "diff")
    ctrl["brent_ret"] = changes_on_calendar(fred["DCOILBRENTEU"], days, "return")
    f = ff.reindex(days)
    f.columns = ["ff_" + c.lower().replace("-", "_") for c in f.columns]
    return ctrl.join(f)

ctrl_eu = controls(EU_DAYS, ff_eu)
ctrl_us = controls(US_DAYS, ff_us)
ctrl_eu.describe().T.round(4)

,count,mean,std,min,25%,50%,75%,max
vix,3869.0,18.4682,6.9608,9.1400,13.6800,16.7200,21.2500,82.6900
vix_chg,3868.0,-0.0009,1.8881,-18.7100,-0.7400,-0.1000,0.5700,24.8600
brent_ret,3880.0,0.0003,0.0265,-0.4747,-0.0106,0.0006,0.0115,0.5099
ff_mkt_rf,3948.0,0.0003,0.0114,-0.1200,-0.0052,0.0006,0.0062,0.0854
ff_smb,3948.0,-0.0000,0.0044,-0.0330,-0.0025,0.0000,0.0025,0.0329
ff_hml,3948.0,0.0000,0.0054,-0.0304,-0.0029,-0.0002,0.0029,0.0438
ff_rf,3948.0,0.0000,0.0001,0.0000,0.0000,0.0000,0.0001,0.0002


## 9. Assemble, check and save

One panel per market, one row per trading day. `stock_returns_eur.csv` keeps the individual stock returns for the leave-one-out robustness check.

In [13]:
panel_eu = pd.DataFrame({
    "spread": spread_eu, "green": green_eu, "brown": brown_eu,
    "n_green": n_green, "n_brown": n_brown,
    "spread_liq_robust": spread_eu_liq_robust, "spread_local_ccy": spread_eu_local,
}, index=EU_DAYS).join(ctrl_eu).join(mccc_eu)

panel_us = pd.DataFrame({
    "spread": spread_us, "icln": ret_us["ICLN"], "xle": ret_us["XLE"], "spy": ret_us["SPY"],
}, index=US_DAYS).join(ctrl_us).join(mccc_us)

panel_eu.index.name = panel_us.index.name = "date"

def check(panel, name):
    core = ["spread", "mccc_Aggregate", "brent_ret", "vix_chg"]
    usable = panel[core].dropna()
    print(f"{name}: {len(panel)} days, {len(usable)} with spread, MCCC, Brent and VIX "
          f"({usable.index.min().date()} -> {usable.index.max().date()})")
    return panel.isna().mean().mul(100).round(1).rename(f"% missing ({name})")

missing = pd.concat([check(panel_eu, "Europe"), check(panel_us, "U.S.")], axis=1)
log["n_usable_days_eu"] = int(panel_eu[["spread", "mccc_Aggregate", "brent_ret", "vix_chg"]].dropna().shape[0])
log["n_usable_days_us"] = int(panel_us[["spread", "mccc_Aggregate", "brent_ret", "vix_chg"]].dropna().shape[0])
missing[missing.max(axis=1) > 0]

Europe: 3948 days, 3839 with spread, MCCC, Brent and VIX (2010-01-05 -> 2025-06-30)
U.S.: 3896 days, 3854 with spread, MCCC, Brent and VIX (2010-01-05 -> 2025-06-30)


,% missing (Europe),% missing (U.S.)
spread_liq_robust,0.1,NaN
vix,2.0,0.0
vix_chg,2.0,0.0
brent_ret,1.7,1.1


In [14]:
# mccc_*_calendar_full.csv: MCCC from 2003 on each calendar, to estimate expected concern in 03_model
panel_eu.to_csv(PROC / "panel_eu.csv")
panel_us.to_csv(PROC / "panel_us.csv")
mccc_eu.to_csv(PROC / "mccc_eu_calendar_full.csv")
mccc_us.to_csv(PROC / "mccc_us_calendar_full.csv")
apply_entry(ret_eur, entry_main).to_csv(PROC / "stock_returns_eur.csv")
entry.to_csv(PROC / "liquidity_entry.csv")
zero_share.to_csv(PROC / "zero_return_share_by_year.csv")
(PROC / "cleaning_log.json").write_text(json.dumps(log, indent=2, default=str))

print("Saved:", sorted(p.name for p in PROC.iterdir()))

Saved: ['cleaning_log.json', 'liquidity_entry.csv', 'mccc_eu_calendar_full.csv', 'mccc_us_calendar_full.csv', 'panel_eu.csv', 'panel_us.csv', 'stock_returns_eur.csv', 'zero_return_share_by_year.csv']
